# 🤖 Device Failure Prediction - Production ML Pipeline

## Overview

This notebook implements an end-to-end ML pipeline for predictive device maintenance, training XGBoost models on operational telemetry and deploying them to Snowflake for real-time inference.

| Pipeline Stage | Snowflake Feature | Output |
|----------------|-------------------|--------|
| **Feature Engineering** | SQL window functions | 29 engineered features |
| **Model Training** | XGBoost classification & regression | Trained sklearn models |
| **Model Registry** | `snowflake.ml.registry.log_model()` | Versioned, governed models |
| **Operationalization** | `MODEL!PREDICT()` in SQL views | Real-time inference |
| **Agent Integration** | Semantic Views | Cortex Agent accessibility |

## Models Produced

| Model | Algorithm | Business Question | Key Predictive Features |
|-------|-----------|-------------------|-------------------------|
| `DEVICE_FAILURE_CLASSIFIER` | XGBoost | Will this device fail in 48h? | TREND features, ERROR_ACCELERATION |
| `DEVICE_HOURS_TO_FAILURE` | XGBoost | How many hours until failure? | Rolling stats + device attributes |
| `LAST_GASP_CLASSIFIER` | RandomForest | Why did this device go offline? | Signal patterns, hardware metrics |

## Architecture

```
Raw Telemetry → Feature Engineering → XGBoost Training → Model Registry → Prediction Table → Agent
     ↓                   ↓                  ↓                ↓                  ↓
DEVICE_TELEMETRY    Window funcs      log_model()     Registry       T_ML_PREDICTIONS
                    TREND features    + metrics       storage        (materialized)
```

## Prerequisites

- SQL scripts 01-05 executed (base tables and data infrastructure)
- 07_expanded_training_data.sql recommended for robust training set

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import pandas as pd
import numpy as np

session = get_active_session()

session.sql("USE DATABASE DEVICE_MAINTENANCE").collect()
session.sql("USE SCHEMA DEVICE_OPS").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

## 1. Data Exploration

> **📊 Production Consideration**: Before training any ML model, rigorous data profiling is essential. We validate data quality, check for distribution shifts, and ensure sufficient volume for reliable model training. In production, this step would include automated data quality checks and drift detection.

### 1.1 Load Source Tables

In [ ]:
inventory_df = session.table("DEVICE_INVENTORY")
telemetry_df = session.table("DEVICE_TELEMETRY")
maintenance_df = session.table("MAINTENANCE_HISTORY")
last_gasp_df = session.table("DEVICE_LAST_GASP")

print("=== Dataset Sizes ===")
print(f"Devices: {inventory_df.count():,}")
print(f"Telemetry records: {telemetry_df.count():,}")
print(f"Maintenance tickets: {maintenance_df.count():,}")
print(f"Last gasp events: {last_gasp_df.count():,}")

In [ ]:
print("=== Device Status Distribution ===")
inventory_df.group_by("STATUS").count().show()

print("\n=== Network Type Distribution ===")
inventory_df.group_by("NETWORK_TYPE").count().show()

print("\n=== Telemetry Sample ===")
telemetry_df.select("DEVICE_ID", "TIMESTAMP", "CPU_TEMP_CELSIUS", "MEMORY_USAGE_PCT", 
                    "WIFI_SIGNAL_STRENGTH", "ERROR_COUNT").limit(5).show()

In [ ]:
print("=== Maintenance Issue Types ===")
maintenance_df.group_by("ISSUE_TYPE").count().order_by(F.col("COUNT").desc()).show()

print("\n=== Last Gasp Classified Causes ===")
last_gasp_df.group_by("CLASSIFIED_CAUSE").count().order_by(F.col("COUNT").desc()).show()

In [ ]:
print("=== Telemetry Statistics ===")
telemetry_df.select(
    F.avg("CPU_TEMP_CELSIUS").alias("avg_cpu_temp"),
    F.avg("MEMORY_USAGE_PCT").alias("avg_memory_pct"),
    F.avg("WIFI_SIGNAL_STRENGTH").alias("avg_wifi_signal"),
    F.avg("ERROR_COUNT").alias("avg_errors"),
    F.stddev("CPU_TEMP_CELSIUS").alias("std_cpu_temp"),
    F.stddev("WIFI_SIGNAL_STRENGTH").alias("std_wifi_signal")
).show()

## 2. Feature Engineering

**Key Features for Device Failure Prediction:**

| Feature Category | Features | Why They Matter |
|-----------------|----------|-----------------|
| **Current State** | CPU_TEMP, CPU_USAGE, MEMORY_PCT, ERROR_COUNT | Snapshot of device health |
| **24h Rolling Stats** | AVG, MAX, MIN over last 24 hours | Smooths noise, captures sustained issues |
| **7-day Baseline** | Longer-term averages | Establishes normal behavior |
| **TREND Features** | CPU_TEMP_TREND, MEMORY_TREND, ERROR_ACCELERATION | **CRITICAL**: Degradation patterns predict failures |
| **Device Attributes** | AGE, DAYS_SINCE_MAINTENANCE, NETWORK_TYPE | Static risk factors |

**19 features** are engineered from raw telemetry for each device:

| Feature Category | Features | Description |
|-----------------|----------|-------------|
| **24-hour metrics** | AVG/MAX CPU temp, memory, errors, Wi-Fi signal | Recent device health |
| **7-day metrics** | AVG CPU temp, memory, errors, Wi-Fi signal | Longer-term baseline |
| **Trend features** | CPU temp trend, Wi-Fi signal trend | Direction of change (24h vs prior 24h) |
| **Volatility** | Wi-Fi signal stddev | Connection stability |
| **Device context** | Age, network type, device type | Static attributes |
| **Maintenance history** | Total tickets, tickets last 30d | Historical reliability |

The SQL below aggregates telemetry using window functions to create these features.

In [ ]:
from snowflake.snowpark.window import Window

device_features_sql = """
WITH hourly_data AS (
    SELECT 
        DEVICE_ID,
        TIMESTAMP,
        CPU_TEMP_CELSIUS,
        CPU_USAGE_PCT,
        MEMORY_USAGE_PCT,
        ERROR_COUNT,
        WIFI_SIGNAL_STRENGTH,
        NETWORK_LATENCY_MS,
        UPTIME_HOURS,
        DISK_USAGE_PCT
    FROM DEVICE_TELEMETRY
),
rolling_stats AS (
    SELECT 
        h.DEVICE_ID,
        h.TIMESTAMP,
        d.DEVICE_MODEL,
        d.NETWORK_TYPE,
        d.INSTALL_DATE,
        d.LAST_MAINTENANCE_DATE,
        
        -- Current metrics
        h.CPU_TEMP_CELSIUS,
        h.CPU_USAGE_PCT,
        h.MEMORY_USAGE_PCT,
        h.ERROR_COUNT,
        h.WIFI_SIGNAL_STRENGTH,
        h.NETWORK_LATENCY_MS,
        h.UPTIME_HOURS,
        
        -- 24-hour rolling averages
        AVG(h.CPU_TEMP_CELSIUS) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as AVG_CPU_TEMP_24H,
        AVG(h.CPU_USAGE_PCT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as AVG_CPU_USAGE_24H,
        AVG(h.MEMORY_USAGE_PCT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as AVG_MEMORY_24H,
        SUM(h.ERROR_COUNT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as ERRORS_24H,
        AVG(h.WIFI_SIGNAL_STRENGTH) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as AVG_WIFI_SIGNAL_24H,
        
        -- 24-hour max values
        MAX(h.CPU_TEMP_CELSIUS) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as MAX_CPU_TEMP_24H,
        MAX(h.CPU_USAGE_PCT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as MAX_CPU_USAGE_24H,
        MAX(h.MEMORY_USAGE_PCT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as MAX_MEMORY_24H,
        MIN(h.WIFI_SIGNAL_STRENGTH) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as MIN_WIFI_SIGNAL_24H,
        
        -- 7-day rolling averages (168 hours)
        AVG(h.CPU_TEMP_CELSIUS) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 168 PRECEDING AND CURRENT ROW) as AVG_CPU_TEMP_7D,
        AVG(h.MEMORY_USAGE_PCT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 168 PRECEDING AND CURRENT ROW) as AVG_MEMORY_7D,
        SUM(h.ERROR_COUNT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 168 PRECEDING AND CURRENT ROW) as ERRORS_7D,
        STDDEV(h.WIFI_SIGNAL_STRENGTH) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 168 PRECEDING AND CURRENT ROW) as WIFI_SIGNAL_VOLATILITY,
        
        -- TREND FEATURES (change from 24h ago to now) - KEY for prediction!
        h.CPU_TEMP_CELSIUS - LAG(h.CPU_TEMP_CELSIUS, 24) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP) as CPU_TEMP_TREND_24H,
        h.CPU_USAGE_PCT - LAG(h.CPU_USAGE_PCT, 24) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP) as CPU_USAGE_TREND_24H,
        h.MEMORY_USAGE_PCT - LAG(h.MEMORY_USAGE_PCT, 24) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP) as MEMORY_TREND_24H,
        h.WIFI_SIGNAL_STRENGTH - LAG(h.WIFI_SIGNAL_STRENGTH, 24) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP) as WIFI_SIGNAL_TREND_24H,
        
        -- ERROR ACCELERATION (change in error rate) - critical for catching degradation
        (SUM(h.ERROR_COUNT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 24 PRECEDING AND CURRENT ROW)) -
        (SUM(h.ERROR_COUNT) OVER (PARTITION BY h.DEVICE_ID ORDER BY h.TIMESTAMP ROWS BETWEEN 48 PRECEDING AND 24 PRECEDING)) as ERROR_ACCELERATION
        
    FROM hourly_data h
    JOIN DEVICE_INVENTORY d ON h.DEVICE_ID = d.DEVICE_ID
)
SELECT 
    DEVICE_ID,
    TIMESTAMP,
    CPU_TEMP_CELSIUS, CPU_USAGE_PCT, MEMORY_USAGE_PCT, ERROR_COUNT, WIFI_SIGNAL_STRENGTH, NETWORK_LATENCY_MS, UPTIME_HOURS,
    ROUND(AVG_CPU_TEMP_24H, 2) as AVG_CPU_TEMP_24H,
    ROUND(AVG_CPU_USAGE_24H, 2) as AVG_CPU_USAGE_24H,
    ROUND(AVG_MEMORY_24H, 2) as AVG_MEMORY_24H,
    ERRORS_24H,
    ROUND(AVG_WIFI_SIGNAL_24H, 2) as AVG_WIFI_SIGNAL_24H,
    ROUND(MAX_CPU_TEMP_24H, 2) as MAX_CPU_TEMP_24H,
    ROUND(MAX_CPU_USAGE_24H, 2) as MAX_CPU_USAGE_24H,
    ROUND(MAX_MEMORY_24H, 2) as MAX_MEMORY_24H,
    MIN_WIFI_SIGNAL_24H,
    ROUND(AVG_CPU_TEMP_7D, 2) as AVG_CPU_TEMP_7D,
    ROUND(AVG_MEMORY_7D, 2) as AVG_MEMORY_7D,
    ERRORS_7D,
    ROUND(COALESCE(WIFI_SIGNAL_VOLATILITY, 0), 2) as WIFI_SIGNAL_VOLATILITY,
    ROUND(COALESCE(CPU_TEMP_TREND_24H, 0), 2) as CPU_TEMP_TREND_24H,
    ROUND(COALESCE(CPU_USAGE_TREND_24H, 0), 2) as CPU_USAGE_TREND_24H,
    ROUND(COALESCE(MEMORY_TREND_24H, 0), 2) as MEMORY_TREND_24H,
    ROUND(COALESCE(WIFI_SIGNAL_TREND_24H, 0), 2) as WIFI_SIGNAL_TREND_24H,
    COALESCE(ERROR_ACCELERATION, 0) as ERROR_ACCELERATION,
    CASE DEVICE_MODEL WHEN 'HealthScreen Pro 55' THEN 0 WHEN 'HealthScreen Lite 32' THEN 1 ELSE 2 END as DEVICE_TYPE_ENCODED,
    CASE NETWORK_TYPE WHEN 'PROVIDER_WIFI' THEN 0 WHEN 'COMPANY_MANAGED' THEN 1 ELSE 2 END as NETWORK_TYPE_ENCODED,
    DATEDIFF('day', INSTALL_DATE, TIMESTAMP) as DEVICE_AGE_DAYS,
    DATEDIFF('day', LAST_MAINTENANCE_DATE, TIMESTAMP) as DAYS_SINCE_MAINTENANCE
FROM rolling_stats
WHERE TIMESTAMP >= DATEADD('day', 7, (SELECT MIN(TIMESTAMP) FROM DEVICE_TELEMETRY))
"""

features_df = session.sql(device_features_sql)
print(f"Feature dataset: {features_df.count()} records")
print(f"\\nFeature columns ({len(features_df.columns)}):")
print([c for c in features_df.columns if c not in ['DEVICE_ID', 'TIMESTAMP']])

### 2.1 Create Training Labels from MAINTENANCE_HISTORY

> **⚠️ Critical Design Decision**: Labels are derived from actual failure events in `MAINTENANCE_HISTORY`, not current device status. This ensures the model learns to predict *future* failures from *current* telemetry patterns.

**Label Engineering Logic:**

For each telemetry timestamp, we look forward in time to determine:
- `WILL_FAIL_48H = 1` if device had a maintenance ticket within the next 48 hours
- `HOURS_TO_FAILURE` = actual hours between telemetry reading and failure event

**Why This Matters:**
- Creates a proper supervised learning problem: features (current telemetry) → outcomes (future failures)
- Captures the temporal relationship between degradation patterns and failure events
- Enables the model to learn early warning signals that precede device failures

**Handling Right-Censored Data:**
- Devices without a failure in the 48h window are labeled `WILL_FAIL_48H = 0`
- `HOURS_TO_FAILURE` defaults to 200 for non-failure cases (represents healthy operating horizon)

In [ ]:
# Create training labels using a JOIN approach (avoids correlated subquery issues)
# First, materialize features to a temp table for performance

session.sql("CREATE OR REPLACE TEMPORARY TABLE TEMP_FEATURES AS " + device_features_sql).collect()
print("Created temporary features table")

# Now create labels by joining with maintenance history
training_labels_sql = """
WITH next_failures AS (
    -- For each device, find maintenance events
    SELECT 
        f.DEVICE_ID,
        f.TIMESTAMP as FEATURE_TIMESTAMP,
        MIN(m.CREATED_AT) as NEXT_FAILURE_TIME
    FROM TEMP_FEATURES f
    LEFT JOIN MAINTENANCE_HISTORY m 
        ON f.DEVICE_ID = m.DEVICE_ID 
        AND m.CREATED_AT > f.TIMESTAMP
        AND m.CREATED_AT <= DATEADD('hour', 48, f.TIMESTAMP)
    GROUP BY f.DEVICE_ID, f.TIMESTAMP
)
SELECT 
    f.*,
    CASE WHEN nf.NEXT_FAILURE_TIME IS NOT NULL THEN 1 ELSE 0 END as WILL_FAIL_48H,
    CASE WHEN nf.NEXT_FAILURE_TIME IS NOT NULL 
         THEN DATEDIFF('hour', f.TIMESTAMP, nf.NEXT_FAILURE_TIME) 
         ELSE 200 
    END as HOURS_TO_FAILURE
FROM TEMP_FEATURES f
LEFT JOIN next_failures nf 
    ON f.DEVICE_ID = nf.DEVICE_ID 
    AND f.TIMESTAMP = nf.FEATURE_TIMESTAMP
"""

training_df = session.sql(training_labels_sql)

print("=== Label Distribution ===")
training_df.group_by("WILL_FAIL_48H").count().show()

training_pandas = training_df.to_pandas()
print(f"\nTraining data shape: {training_pandas.shape}")
print(f"Positive labels (failures): {(training_pandas['WILL_FAIL_48H'] == 1).sum()}")
print(f"Negative labels (no failure): {(training_pandas['WILL_FAIL_48H'] == 0).sum()}")
print(f"Class balance: {(training_pandas['WILL_FAIL_48H'] == 1).sum() / len(training_pandas) * 100:.1f}% positive")

## 3. Model Training with XGBoost

> **🎯 Algorithm Selection Rationale**: XGBoost is the industry standard for tabular prediction tasks, consistently winning Kaggle competitions and powering production systems at scale. Its gradient boosting approach excels at capturing non-linear relationships in sensor data.

**Why XGBoost for Predictive Maintenance?**
| Capability | Benefit for Device Failure Prediction |
|------------|--------------------------------------|
| Feature importance via SHAP | Explainability: "Why is this device at risk?" |
| `scale_pos_weight` parameter | Handles class imbalance (failures are rare events) |
| Regularization (L1/L2) | Prevents overfitting on noisy telemetry data |
| Missing value handling | Robust to sensor dropouts and null readings |
| Fast inference | Sub-millisecond predictions for real-time scoring |

**Model Portfolio:**
1. **Binary Classification**: Will device fail within 48 hours? (prioritization)
2. **Regression**: How many hours until failure? (scheduling)
3. **Multi-class Classification**: Why did device go offline? (root cause)

### 3.1 Classification Model: Will Fail in 48 Hours?

**Hyperparameters:**
- `n_estimators=100`: Balanced between accuracy and training time
- `max_depth=6`: Prevents overfitting while capturing complex patterns
- `scale_pos_weight`: Automatically computed from class imbalance ratio
- `learning_rate=0.1`: Standard rate for production stability

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

feature_cols = [
    'CPU_TEMP_CELSIUS', 'CPU_USAGE_PCT', 'MEMORY_USAGE_PCT', 'ERROR_COUNT', 
    'WIFI_SIGNAL_STRENGTH', 'NETWORK_LATENCY_MS', 'UPTIME_HOURS',
    'AVG_CPU_TEMP_24H', 'AVG_CPU_USAGE_24H', 'AVG_MEMORY_24H', 'ERRORS_24H', 'AVG_WIFI_SIGNAL_24H',
    'MAX_CPU_TEMP_24H', 'MAX_CPU_USAGE_24H', 'MAX_MEMORY_24H', 'MIN_WIFI_SIGNAL_24H',
    'AVG_CPU_TEMP_7D', 'AVG_MEMORY_7D', 'ERRORS_7D', 'WIFI_SIGNAL_VOLATILITY',
    'CPU_TEMP_TREND_24H', 'CPU_USAGE_TREND_24H', 'MEMORY_TREND_24H', 'WIFI_SIGNAL_TREND_24H',
    'ERROR_ACCELERATION',
    'DEVICE_TYPE_ENCODED', 'NETWORK_TYPE_ENCODED', 'DEVICE_AGE_DAYS', 'DAYS_SINCE_MAINTENANCE'
]

df = training_pandas.copy()
for col in feature_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median() if df[col].dtype in ['float64', 'int64'] else 0)

X = df[feature_cols].values
y_class = df['WILL_FAIL_48H'].values
y_reg = df['HOURS_TO_FAILURE'].values

X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

scale_pos_weight = (y_train_class == 0).sum() / max((y_train_class == 1).sum(), 1)
print(f"Train size: {len(X_train)} ({y_train_class.sum()} positive)")
print(f"Test size: {len(X_test)} ({y_test_class.sum()} positive)")
print(f"Class imbalance ratio: {scale_pos_weight:.1f}:1")

In [ ]:
clf_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42
)
clf_model.fit(X_train, y_train_class, eval_set=[(X_test, y_test_class)], verbose=False)

y_pred_class = clf_model.predict(X_test)
y_pred_proba = clf_model.predict_proba(X_test)[:, 1]

clf_metrics = {
    "accuracy": float(accuracy_score(y_test_class, y_pred_class)),
    "precision": float(precision_score(y_test_class, y_pred_class, zero_division=0)),
    "recall": float(recall_score(y_test_class, y_pred_class, zero_division=0)),
    "f1_score": float(f1_score(y_test_class, y_pred_class, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test_class, y_pred_proba)) if len(np.unique(y_test_class)) > 1 else 0.0
}

print("=== XGBoost Classification Model Metrics ===")
for metric, value in clf_metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': clf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== XGBoost Feature Importance (Top 10) ===")
print("These features are most predictive of device failures:\\n")
for i, row in feature_importance.head(10).iterrows():
    bar = "█" * int(row['importance'] * 50)
    print(f"{row['feature']:25s} {row['importance']:.3f} {bar}")

print("\\n💡 KEY INSIGHT: TREND features (CPU_TEMP_TREND, MEMORY_TREND, ERROR_ACCELERATION)")
print("   are highly predictive because they capture DEGRADATION patterns before failure!")

### 3.2 Regression Model: Hours to Failure

**Algorithm:** XGBoost Regressor
- `n_estimators=100`, `max_depth=6`, `learning_rate=0.1`
- `objective='reg:squarederror'` for continuous output
- Predicts hours until failure for maintenance scheduling

In [ ]:
reg_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='reg:squarederror',
    random_state=42
)
reg_model.fit(X_train, y_train_reg, eval_set=[(X_test, y_test_reg)], verbose=False)

y_pred_reg = reg_model.predict(X_test)

reg_metrics = {
    "mae": float(mean_absolute_error(y_test_reg, y_pred_reg)),
    "rmse": float(np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))),
    "r2_score": float(r2_score(y_test_reg, y_pred_reg))
}

print("=== XGBoost Regression Model Metrics (Hours to Failure) ===")
for metric, value in reg_metrics.items():
    print(f"{metric}: {value:.4f}")

### 3.3 Last Gasp Classification Model

**Purpose:** Classify WHY a device went offline using its final telemetry readings.

**Classes:**
- `WIFI_PASSWORD_CHANGE` - Sudden signal drop, healthy metrics → Call office
- `HARDWARE_FAILURE` - High CPU/errors, stable signal → Dispatch technician  
- `NETWORK_OUTAGE` - Gradual decline, multiple devices → Wait and monitor
- `POWER_LOSS` - All metrics normal, instant disconnect → Remote restart

**Features:** Last signal strength, CPU temp, memory %, error count, signal trend, drop rate

In [ ]:
last_gasp_sql = """
SELECT 
    LAST_SIGNAL_STRENGTH,
    LAST_CPU_TEMP,
    LAST_MEMORY_PCT,
    LAST_ERROR_COUNT,
    CASE SIGNAL_TREND 
        WHEN 'SUDDEN_DROP' THEN 0 
        WHEN 'GRADUAL_DECLINE' THEN 1 
        ELSE 2 
    END as SIGNAL_TREND_ENCODED,
    SIGNAL_DROP_RATE,
    CLASSIFIED_CAUSE
FROM DEVICE_LAST_GASP
WHERE CLASSIFIED_CAUSE IS NOT NULL
"""

last_gasp_pandas = session.sql(last_gasp_sql).to_pandas()
print(f"Last gasp training data: {len(last_gasp_pandas)} records")
print("\n=== Cause Distribution ===")
print(last_gasp_pandas['CLASSIFIED_CAUSE'].value_counts())

In [ ]:
if len(last_gasp_pandas) > 10:
    lg_features = ['LAST_SIGNAL_STRENGTH', 'LAST_CPU_TEMP', 'LAST_MEMORY_PCT', 
                   'LAST_ERROR_COUNT', 'SIGNAL_TREND_ENCODED', 'SIGNAL_DROP_RATE']
    
    lg_df = last_gasp_pandas.copy()
    for col in lg_features:
        lg_df[col] = lg_df[col].fillna(lg_df[col].median() if lg_df[col].dtype in ['float64', 'int64'] else 0)
    
    cause_encoder = LabelEncoder()
    lg_df['CAUSE_ENCODED'] = cause_encoder.fit_transform(lg_df['CLASSIFIED_CAUSE'])
    cause_classes = cause_encoder.classes_
    
    X_lg = lg_df[lg_features].values
    y_lg = lg_df['CAUSE_ENCODED'].values
    
    X_lg_train, X_lg_test, y_lg_train, y_lg_test = train_test_split(X_lg, y_lg, test_size=0.2, random_state=42)
    
    last_gasp_model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
    last_gasp_model.fit(X_lg_train, y_lg_train)
    
    y_lg_pred = last_gasp_model.predict(X_lg_test)
    lg_accuracy = accuracy_score(y_lg_test, y_lg_pred)
    
    print(f"Last Gasp Classification Accuracy: {lg_accuracy:.4f}")
    print(f"Classes: {list(cause_classes)}")
else:
    print("Insufficient last gasp data for ML training - using rule-based classification")
    last_gasp_model = None
    cause_classes = None

## 4. Model Registry - Log Models

> **🔒 MLOps Best Practice**: The Snowflake Model Registry provides enterprise-grade model governance. Every model is versioned, auditable, and accessible via SQL—eliminating the "model serialization hell" of traditional ML deployment.

**Why Model Registry Matters in Production:**

| Challenge | How Registry Solves It |
|-----------|------------------------|
| Model versioning | Automatic version tracking with rollback capability |
| Reproducibility | Metrics, dependencies, and signatures stored with model |
| Access control | RBAC through Snowflake's native security model |
| Deployment friction | `MODEL!PREDICT()` syntax—no API servers required |
| Audit compliance | Full lineage from training data to predictions |

**The `log_model()` call:**
1. Serializes the sklearn/XGBoost model to Snowflake-native format
2. Creates a MODEL object in the current schema (first-class DB citizen)
3. Records evaluation metrics for model comparison and A/B testing
4. Captures input/output schema for type-safe inference

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints
from snowflake.ml.model.model_signature import FeatureSpec, DataType, ModelSignature

registry = Registry(session=session, database_name="DEVICE_MAINTENANCE", schema_name="DEVICE_OPS")
print("Registry initialized")

In [ ]:
sample_input = pd.DataFrame(X_train[:5], columns=feature_cols)

clf_version = registry.log_model(
    clf_model,
    model_name="DEVICE_FAILURE_CLASSIFIER",
    version_name="V1",
    comment="Binary classification: Will device fail within 48 hours?",
    metrics=clf_metrics,
    sample_input_data=sample_input
)

print(f"Classification model logged: {clf_version.model_name} v{clf_version.version_name}")

In [ ]:
reg_version = registry.log_model(
    reg_model,
    model_name="DEVICE_HOURS_TO_FAILURE",
    version_name="V1",
    comment="Regression: Predicted hours until device failure",
    metrics=reg_metrics,
    sample_input_data=sample_input
)

print(f"Regression model logged: {reg_version.model_name} v{reg_version.version_name}")

In [ ]:
if last_gasp_model is not None:
    lg_sample = pd.DataFrame(X_lg_train[:5], columns=lg_features)
    
    lg_version = registry.log_model(
        last_gasp_model,
        model_name="LAST_GASP_CLASSIFIER",
        version_name="V1",
        comment="Classifies offline cause: WIFI_PASSWORD_CHANGE, HARDWARE_FAILURE, NETWORK_OUTAGE, POWER_LOSS",
        metrics={"accuracy": float(lg_accuracy)},
        sample_input_data=lg_sample
    )
    print(f"Last gasp model logged: {lg_version.model_name} v{lg_version.version_name}")
else:
    print("Last gasp model not trained - skipping registry")

In [ ]:
print("=== Registered Models ===")
registry.show_models()

## 5. Batch Inference - Create Prediction Views

> **⚡ Operationalization Strategy**: The feature engineering logic must be identical between training and inference. By encoding features in SQL views, we guarantee consistency and enable real-time scoring without Python infrastructure.

### 5.1 Feature View

**`V_DEVICE_ML_FEATURES`** - The Feature Store Pattern

This view computes real-time features from live telemetry data, mirroring the exact transformations used during training:
- **Rolling window aggregations**: 24-hour and 7-day statistics
- **Trend calculations**: Rate of change in key metrics
- **Device context**: Age, maintenance history, network type

**Why a View (not a Table)?**
- Always reflects current telemetry state
- No ETL jobs to maintain
- Snowflake's query optimizer handles performance

In [ ]:
create_feature_view_sql = """
CREATE OR REPLACE VIEW V_DEVICE_ML_FEATURES AS
WITH latest_telemetry AS (
    SELECT 
        DEVICE_ID,
        CPU_TEMP_CELSIUS,
        CPU_USAGE_PCT,
        MEMORY_USAGE_PCT,
        ERROR_COUNT,
        WIFI_SIGNAL_STRENGTH,
        NETWORK_LATENCY_MS,
        UPTIME_HOURS,
        TIMESTAMP,
        ROW_NUMBER() OVER (PARTITION BY DEVICE_ID ORDER BY TIMESTAMP DESC) as rn
    FROM DEVICE_TELEMETRY
    WHERE TIMESTAMP > DATEADD('day', -7, CURRENT_TIMESTAMP())
),
rolling_24h AS (
    SELECT 
        DEVICE_ID,
        AVG(CPU_TEMP_CELSIUS) as AVG_CPU_TEMP_24H,
        AVG(CPU_USAGE_PCT) as AVG_CPU_USAGE_24H,
        AVG(MEMORY_USAGE_PCT) as AVG_MEMORY_24H,
        SUM(ERROR_COUNT) as ERRORS_24H,
        AVG(WIFI_SIGNAL_STRENGTH) as AVG_WIFI_SIGNAL_24H,
        MAX(CPU_TEMP_CELSIUS) as MAX_CPU_TEMP_24H,
        MAX(CPU_USAGE_PCT) as MAX_CPU_USAGE_24H,
        MAX(MEMORY_USAGE_PCT) as MAX_MEMORY_24H,
        MIN(WIFI_SIGNAL_STRENGTH) as MIN_WIFI_SIGNAL_24H
    FROM DEVICE_TELEMETRY
    WHERE TIMESTAMP > DATEADD('hour', -24, CURRENT_TIMESTAMP())
    GROUP BY DEVICE_ID
),
rolling_7d AS (
    SELECT 
        DEVICE_ID,
        AVG(CPU_TEMP_CELSIUS) as AVG_CPU_TEMP_7D,
        AVG(MEMORY_USAGE_PCT) as AVG_MEMORY_7D,
        SUM(ERROR_COUNT) as ERRORS_7D,
        STDDEV(WIFI_SIGNAL_STRENGTH) as WIFI_SIGNAL_VOLATILITY
    FROM DEVICE_TELEMETRY
    WHERE TIMESTAMP > DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY DEVICE_ID
),
prev_24h AS (
    SELECT 
        DEVICE_ID,
        AVG(CPU_TEMP_CELSIUS) as PREV_CPU_TEMP,
        AVG(CPU_USAGE_PCT) as PREV_CPU_USAGE,
        AVG(MEMORY_USAGE_PCT) as PREV_MEMORY,
        AVG(WIFI_SIGNAL_STRENGTH) as PREV_WIFI_SIGNAL,
        SUM(ERROR_COUNT) as PREV_ERRORS
    FROM DEVICE_TELEMETRY
    WHERE TIMESTAMP BETWEEN DATEADD('hour', -48, CURRENT_TIMESTAMP()) AND DATEADD('hour', -24, CURRENT_TIMESTAMP())
    GROUP BY DEVICE_ID
)
SELECT 
    d.DEVICE_ID,
    d.DEVICE_MODEL,
    d.FACILITY_NAME,
    d.STATUS,
    t.CPU_TEMP_CELSIUS,
    t.CPU_USAGE_PCT,
    t.MEMORY_USAGE_PCT,
    t.ERROR_COUNT,
    t.WIFI_SIGNAL_STRENGTH,
    t.NETWORK_LATENCY_MS,
    t.UPTIME_HOURS,
    ROUND(r24.AVG_CPU_TEMP_24H, 2) as AVG_CPU_TEMP_24H,
    ROUND(r24.AVG_CPU_USAGE_24H, 2) as AVG_CPU_USAGE_24H,
    ROUND(r24.AVG_MEMORY_24H, 2) as AVG_MEMORY_24H,
    r24.ERRORS_24H,
    ROUND(r24.AVG_WIFI_SIGNAL_24H, 2) as AVG_WIFI_SIGNAL_24H,
    ROUND(r24.MAX_CPU_TEMP_24H, 2) as MAX_CPU_TEMP_24H,
    ROUND(r24.MAX_CPU_USAGE_24H, 2) as MAX_CPU_USAGE_24H,
    ROUND(r24.MAX_MEMORY_24H, 2) as MAX_MEMORY_24H,
    r24.MIN_WIFI_SIGNAL_24H,
    ROUND(r7.AVG_CPU_TEMP_7D, 2) as AVG_CPU_TEMP_7D,
    ROUND(r7.AVG_MEMORY_7D, 2) as AVG_MEMORY_7D,
    r7.ERRORS_7D,
    ROUND(COALESCE(r7.WIFI_SIGNAL_VOLATILITY, 0), 2) as WIFI_SIGNAL_VOLATILITY,
    -- TREND FEATURES
    ROUND(COALESCE(r24.AVG_CPU_TEMP_24H - p.PREV_CPU_TEMP, 0), 2) as CPU_TEMP_TREND_24H,
    ROUND(COALESCE(r24.AVG_CPU_USAGE_24H - p.PREV_CPU_USAGE, 0), 2) as CPU_USAGE_TREND_24H,
    ROUND(COALESCE(r24.AVG_MEMORY_24H - p.PREV_MEMORY, 0), 2) as MEMORY_TREND_24H,
    ROUND(COALESCE(r24.AVG_WIFI_SIGNAL_24H - p.PREV_WIFI_SIGNAL, 0), 2) as WIFI_SIGNAL_TREND_24H,
    COALESCE(r24.ERRORS_24H - p.PREV_ERRORS, 0) as ERROR_ACCELERATION,
    -- Device attributes
    CASE d.DEVICE_MODEL WHEN 'HealthScreen Pro 55' THEN 0 WHEN 'HealthScreen Lite 32' THEN 1 ELSE 2 END as DEVICE_TYPE_ENCODED,
    CASE d.NETWORK_TYPE WHEN 'PROVIDER_WIFI' THEN 0 WHEN 'COMPANY_MANAGED' THEN 1 ELSE 2 END as NETWORK_TYPE_ENCODED,
    DATEDIFF('day', d.INSTALL_DATE, CURRENT_DATE()) as DEVICE_AGE_DAYS,
    DATEDIFF('day', d.LAST_MAINTENANCE_DATE, CURRENT_DATE()) as DAYS_SINCE_MAINTENANCE
FROM DEVICE_INVENTORY d
LEFT JOIN latest_telemetry t ON d.DEVICE_ID = t.DEVICE_ID AND t.rn = 1
LEFT JOIN rolling_24h r24 ON d.DEVICE_ID = r24.DEVICE_ID
LEFT JOIN rolling_7d r7 ON d.DEVICE_ID = r7.DEVICE_ID
LEFT JOIN prev_24h p ON d.DEVICE_ID = p.DEVICE_ID
"""

session.sql(create_feature_view_sql).collect()
print("Created V_DEVICE_ML_FEATURES view with enhanced features")

### 5.2 Prediction View - Real-Time ML Inference

> **🚀 The Power of `MODEL!PREDICT()`**: This is where ML becomes operationally useful. The view below invokes trained XGBoost models directly in SQL—no Python servers, no API calls, no serialization overhead.

**Inference Architecture:**

```sql
DEVICE_FAILURE_CLASSIFIER!PREDICT(feature1, feature2, ...):output_feature_0
```

**Execution Flow:**
1. `V_DEVICE_ML_FEATURES` computes live features from current telemetry
2. `DEVICE_FAILURE_CLASSIFIER!PREDICT()` invokes the XGBoost classifier from the registry
3. `DEVICE_HOURS_TO_FAILURE!PREDICT()` invokes the XGBoost regressor
4. Results are returned as view columns, joinable with any SQL query

**Production Benefits:**
- **Real-time**: Every query runs fresh inference on current data
- **Scalable**: Snowflake's compute handles any volume
- **Governed**: Same access controls as any other view
- **Observable**: Query history tracks all inference calls

In [ ]:
create_prediction_view_sql = """
CREATE OR REPLACE VIEW V_ML_FAILURE_PREDICTIONS AS
WITH feature_data AS (
    SELECT 
        DEVICE_ID,
        DEVICE_MODEL,
        FACILITY_NAME,
        STATUS,
        CPU_TEMP_CELSIUS, CPU_USAGE_PCT, MEMORY_USAGE_PCT, ERROR_COUNT,
        WIFI_SIGNAL_STRENGTH, NETWORK_LATENCY_MS, UPTIME_HOURS,
        AVG_CPU_TEMP_24H, AVG_CPU_USAGE_24H, AVG_MEMORY_24H, ERRORS_24H, AVG_WIFI_SIGNAL_24H,
        MAX_CPU_TEMP_24H, MAX_CPU_USAGE_24H, MAX_MEMORY_24H, MIN_WIFI_SIGNAL_24H,
        AVG_CPU_TEMP_7D, AVG_MEMORY_7D, ERRORS_7D, WIFI_SIGNAL_VOLATILITY,
        CPU_TEMP_TREND_24H, CPU_USAGE_TREND_24H, MEMORY_TREND_24H, WIFI_SIGNAL_TREND_24H, ERROR_ACCELERATION,
        DEVICE_TYPE_ENCODED, NETWORK_TYPE_ENCODED, DEVICE_AGE_DAYS, DAYS_SINCE_MAINTENANCE
    FROM V_DEVICE_ML_FEATURES
)
SELECT 
    f.DEVICE_ID,
    f.DEVICE_MODEL,
    f.FACILITY_NAME,
    f.STATUS,
    -- XGBoost Classification: Will fail within 48h?
    DEVICE_FAILURE_CLASSIFIER!PREDICT(
        f.CPU_TEMP_CELSIUS, f.CPU_USAGE_PCT, f.MEMORY_USAGE_PCT, f.ERROR_COUNT,
        f.WIFI_SIGNAL_STRENGTH, f.NETWORK_LATENCY_MS, f.UPTIME_HOURS,
        f.AVG_CPU_TEMP_24H, f.AVG_CPU_USAGE_24H, f.AVG_MEMORY_24H, f.ERRORS_24H, f.AVG_WIFI_SIGNAL_24H,
        f.MAX_CPU_TEMP_24H, f.MAX_CPU_USAGE_24H, f.MAX_MEMORY_24H, f.MIN_WIFI_SIGNAL_24H,
        f.AVG_CPU_TEMP_7D, f.AVG_MEMORY_7D, f.ERRORS_7D, f.WIFI_SIGNAL_VOLATILITY,
        f.CPU_TEMP_TREND_24H, f.CPU_USAGE_TREND_24H, f.MEMORY_TREND_24H, f.WIFI_SIGNAL_TREND_24H, f.ERROR_ACCELERATION,
        f.DEVICE_TYPE_ENCODED, f.NETWORK_TYPE_ENCODED, f.DEVICE_AGE_DAYS, f.DAYS_SINCE_MAINTENANCE
    ):output_feature_0::INT as WILL_FAIL_48H,
    -- XGBoost Regression: Hours until failure
    ROUND(DEVICE_HOURS_TO_FAILURE!PREDICT(
        f.CPU_TEMP_CELSIUS, f.CPU_USAGE_PCT, f.MEMORY_USAGE_PCT, f.ERROR_COUNT,
        f.WIFI_SIGNAL_STRENGTH, f.NETWORK_LATENCY_MS, f.UPTIME_HOURS,
        f.AVG_CPU_TEMP_24H, f.AVG_CPU_USAGE_24H, f.AVG_MEMORY_24H, f.ERRORS_24H, f.AVG_WIFI_SIGNAL_24H,
        f.MAX_CPU_TEMP_24H, f.MAX_CPU_USAGE_24H, f.MAX_MEMORY_24H, f.MIN_WIFI_SIGNAL_24H,
        f.AVG_CPU_TEMP_7D, f.AVG_MEMORY_7D, f.ERRORS_7D, f.WIFI_SIGNAL_VOLATILITY,
        f.CPU_TEMP_TREND_24H, f.CPU_USAGE_TREND_24H, f.MEMORY_TREND_24H, f.WIFI_SIGNAL_TREND_24H, f.ERROR_ACCELERATION,
        f.DEVICE_TYPE_ENCODED, f.NETWORK_TYPE_ENCODED, f.DEVICE_AGE_DAYS, f.DAYS_SINCE_MAINTENANCE
    ):output_feature_0::FLOAT, 1) as PREDICTED_HOURS_TO_FAILURE,
    -- Risk level based on ML prediction + trend signals
    CASE 
        WHEN DEVICE_FAILURE_CLASSIFIER!PREDICT(
            f.CPU_TEMP_CELSIUS, f.CPU_USAGE_PCT, f.MEMORY_USAGE_PCT, f.ERROR_COUNT,
            f.WIFI_SIGNAL_STRENGTH, f.NETWORK_LATENCY_MS, f.UPTIME_HOURS,
            f.AVG_CPU_TEMP_24H, f.AVG_CPU_USAGE_24H, f.AVG_MEMORY_24H, f.ERRORS_24H, f.AVG_WIFI_SIGNAL_24H,
            f.MAX_CPU_TEMP_24H, f.MAX_CPU_USAGE_24H, f.MAX_MEMORY_24H, f.MIN_WIFI_SIGNAL_24H,
            f.AVG_CPU_TEMP_7D, f.AVG_MEMORY_7D, f.ERRORS_7D, f.WIFI_SIGNAL_VOLATILITY,
            f.CPU_TEMP_TREND_24H, f.CPU_USAGE_TREND_24H, f.MEMORY_TREND_24H, f.WIFI_SIGNAL_TREND_24H, f.ERROR_ACCELERATION,
            f.DEVICE_TYPE_ENCODED, f.NETWORK_TYPE_ENCODED, f.DEVICE_AGE_DAYS, f.DAYS_SINCE_MAINTENANCE
        ):output_feature_0 = 1 THEN 'CRITICAL'
        WHEN f.ERROR_ACCELERATION > 5 OR f.CPU_TEMP_TREND_24H > 10 OR f.MEMORY_TREND_24H > 15 THEN 'WARNING'
        WHEN f.ERRORS_24H > 5 OR f.AVG_WIFI_SIGNAL_24H < -75 THEN 'CAUTION'
        ELSE 'HEALTHY'
    END as RISK_LEVEL,
    -- Key contributing factors for explainability
    CASE 
        WHEN f.CPU_TEMP_TREND_24H > 10 THEN 'Rising CPU temperature (+' || ROUND(f.CPU_TEMP_TREND_24H, 1) || '°C in 24h)'
        WHEN f.MEMORY_TREND_24H > 15 THEN 'Memory pressure increasing (+' || ROUND(f.MEMORY_TREND_24H, 1) || '% in 24h)'
        WHEN f.ERROR_ACCELERATION > 5 THEN 'Error rate accelerating (+' || f.ERROR_ACCELERATION || ' errors)'
        WHEN f.WIFI_SIGNAL_TREND_24H < -10 THEN 'WiFi signal degrading (' || ROUND(f.WIFI_SIGNAL_TREND_24H, 1) || ' dBm)'
        ELSE 'Stable metrics'
    END as PRIMARY_RISK_FACTOR,
    CURRENT_TIMESTAMP() as PREDICTION_TIMESTAMP
FROM feature_data f
"""

session.sql(create_prediction_view_sql).collect()
print("Created V_ML_FAILURE_PREDICTIONS view with XGBoost models")

In [ ]:
print("=== Sample ML Predictions ===")
session.sql("SELECT * FROM V_ML_FAILURE_PREDICTIONS LIMIT 10").show()

In [ ]:
print("=== Prediction Summary ===")
session.sql("""
SELECT 
    RISK_LEVEL,
    COUNT(*) as DEVICE_COUNT,
    ROUND(AVG(PREDICTED_HOURS_TO_FAILURE), 1) as AVG_HOURS_TO_FAILURE
FROM V_ML_FAILURE_PREDICTIONS
GROUP BY RISK_LEVEL
ORDER BY CASE RISK_LEVEL WHEN 'CRITICAL' THEN 1 WHEN 'WARNING' THEN 2 ELSE 3 END
""").show()

### 5.3 Production Operationalization

> **🏭 Production Pattern**: Operationalizing ML models requires automated, scheduled inference on new data. Snowflake provides three mechanisms: **Dynamic Tables**, **Tasks**, and **Stored Procedures**.

**Operationalization Options:**

| Approach | When to Use | Refresh Control |
|----------|-------------|-----------------|
| **Dynamic Table** | Continuous refresh, declarative | `TARGET_LAG` (e.g., 1 hour) |
| **Task + Table** | Precise scheduling, cost control | CRON expression |
| **Stored Procedure** | Complex logic, conditional runs | Called by Task or manually |

**Our Production Architecture:**

```
                    ┌─────────────────────────────────────────┐
                    │         DYNAMIC TABLE                    │
                    │      T_ML_PREDICTIONS_LIVE              │
   New Telemetry    │  ┌─────────────────────────────────┐   │
        ↓           │  │  V_DEVICE_ML_FEATURES (features) │   │
  DEVICE_TELEMETRY ─┼──│            ↓                     │   │
                    │  │  MODEL!PREDICT() (inference)     │   │
                    │  └─────────────────────────────────┘   │
                    │       Auto-refresh every 1 hour         │
                    └─────────────────────────────────────────┘
                                      ↓
                    ┌─────────────────────────────────────────┐
                    │    Downstream Consumers                  │
                    │  • Cortex Agent (natural language)      │
                    │  • Dashboards (BI tools)                │
                    │  • Alerting systems                     │
                    └─────────────────────────────────────────┘
```

In [ ]:
# =============================================================================
# OPTION 1: DYNAMIC TABLE (Recommended for production)
# =============================================================================
# Dynamic Tables automatically refresh based on TARGET_LAG
# No tasks or scheduling required - Snowflake manages the refresh

dynamic_table_sql = """
CREATE OR REPLACE DYNAMIC TABLE T_ML_PREDICTIONS_LIVE
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
SELECT * FROM V_ML_FAILURE_PREDICTIONS
"""

session.sql(dynamic_table_sql).collect()
print("✅ Created DYNAMIC TABLE T_ML_PREDICTIONS_LIVE")
print("   → Auto-refreshes within 1 hour of source data changes")
print("   → No manual scheduling required")

# =============================================================================
# OPTION 2: TASK + STORED PROCEDURE (For complex logic or cost control)
# =============================================================================
# Use when you need precise scheduling or complex refresh logic

stored_proc_sql = """
CREATE OR REPLACE PROCEDURE SP_REFRESH_ML_PREDICTIONS()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN
    -- Log refresh start
    INSERT INTO PREDICTION_REFRESH_LOG (REFRESH_TIME, STATUS, RECORD_COUNT)
    SELECT CURRENT_TIMESTAMP(), 'STARTED', 0;
    
    -- Refresh predictions table with fresh inference
    CREATE OR REPLACE TABLE T_ML_PREDICTIONS AS
    SELECT * FROM V_ML_FAILURE_PREDICTIONS;
    
    -- Log completion with count
    LET prediction_count INT := (SELECT COUNT(*) FROM T_ML_PREDICTIONS);
    
    UPDATE PREDICTION_REFRESH_LOG 
    SET STATUS = 'COMPLETED', RECORD_COUNT = :prediction_count
    WHERE STATUS = 'STARTED';
    
    RETURN 'Refreshed ' || :prediction_count || ' predictions';
END;
$$
"""

# Create log table for audit trail
session.sql("""
CREATE TABLE IF NOT EXISTS PREDICTION_REFRESH_LOG (
    REFRESH_TIME TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    STATUS STRING,
    RECORD_COUNT INT
)
""").collect()

session.sql(stored_proc_sql).collect()
print("\n✅ Created STORED PROCEDURE SP_REFRESH_ML_PREDICTIONS()")

# Create scheduled task
task_sql = """
CREATE OR REPLACE TASK TASK_REFRESH_PREDICTIONS
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = 'USING CRON 0 * * * * UTC'  -- Every hour on the hour
    COMMENT = 'Hourly refresh of ML predictions table'
AS
    CALL SP_REFRESH_ML_PREDICTIONS()
"""

session.sql(task_sql).collect()
print("✅ Created TASK TASK_REFRESH_PREDICTIONS (hourly schedule)")
print("   → Run: ALTER TASK TASK_REFRESH_PREDICTIONS RESUME; to activate")

# =============================================================================
# OPTION 3: IMMEDIATE REFRESH (For initial load / manual refresh)
# =============================================================================

session.sql("CREATE OR REPLACE TABLE T_ML_PREDICTIONS AS SELECT * FROM V_ML_FAILURE_PREDICTIONS").collect()
count = session.sql("SELECT COUNT(*) as cnt FROM T_ML_PREDICTIONS").collect()[0]['CNT']
print(f"\n✅ Created TABLE T_ML_PREDICTIONS with {count} predictions (initial load)")

# Summary
print("\n" + "="*70)
print("PRODUCTION OPERATIONALIZATION COMPLETE")
print("="*70)
print("""
Objects Created:
  • T_ML_PREDICTIONS_LIVE  - Dynamic Table (auto-refresh, recommended)
  • T_ML_PREDICTIONS       - Standard Table (manual/task refresh)
  • SP_REFRESH_ML_PREDICTIONS() - Stored Procedure for refresh logic
  • TASK_REFRESH_PREDICTIONS    - Scheduled task (run RESUME to activate)
  • PREDICTION_REFRESH_LOG      - Audit table for refresh history

To activate scheduled task:
  ALTER TASK TASK_REFRESH_PREDICTIONS RESUME;

To check task status:
  SHOW TASKS LIKE 'TASK_REFRESH%';
  SELECT * FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY()) ORDER BY SCHEDULED_TIME DESC;
""")

## 6. Verify ML Objects Created

The ML pipeline is now complete. The semantic views are consolidated in `02_create_semantic_views.sql`:

| Semantic View | Tables Included | Purpose |
|---------------|-----------------|----------|
| `SV_DEVICE_ANALYTICS` | devices, **T_ML_PREDICTIONS**, last_gasp, downtime | Device health + **ML predictions** + failure classification |
| `SV_MAINTENANCE_OPERATIONS` | work_orders, tickets, technicians, actions | Service & operations |
| `SV_BUSINESS_IMPACT` | revenue, satisfaction, roi | Financial impact |

This consolidation follows Snowflake best practices:
- 3-5 tables per semantic view (star schema pattern)
- RELATIONSHIPS clause for proper joins
- Fewer views = better LLM accuracy


In [ ]:
# NOTE: Semantic views are now consolidated in 02_create_semantic_views.sql
# SV_DEVICE_ANALYTICS includes device health, ML predictions, last gasp, and downtime
# The T_ML_PREDICTIONS table created above is referenced by that unified semantic view

print("="*60)
print("ML PIPELINE COMPLETE!")
print("="*60)
print()
print("Objects created:")
print("  - DEVICE_FAILURE_CLASSIFIER (XGBoost model in registry)")
print("  - DEVICE_HOURS_TO_FAILURE (XGBoost model in registry)")
print("  - V_DEVICE_ML_FEATURES (feature engineering view)")
print("  - V_ML_FAILURE_PREDICTIONS (live inference view)")
print("  - T_ML_PREDICTIONS (pre-computed predictions table)")
print()
print("Semantic views (created by 02_create_semantic_views.sql):")
print("  - SV_DEVICE_ANALYTICS (includes predictions, last gasp, downtime)")
print("  - SV_MAINTENANCE_OPERATIONS (work orders, tickets, technicians)")
print("  - SV_BUSINESS_IMPACT (revenue, satisfaction, ROI)")
print()
print("Next step: Run 06_enhanced_capabilities.sql")


## Summary

### Production ML Pipeline Complete

This notebook delivers a fully operationalized predictive maintenance system:

| Model | Algorithm | Business Value |
|-------|-----------|----------------|
| `DEVICE_FAILURE_CLASSIFIER` | XGBoost Binary | Prioritize devices for proactive intervention |
| `DEVICE_HOURS_TO_FAILURE` | XGBoost Regression | Schedule technician dispatch optimally |
| `LAST_GASP_CLASSIFIER` | RandomForest Multi-class | Route failures to correct resolution path |

### Technical Architecture

| Component | Type | Role in Pipeline |
|-----------|------|------------------|
| `V_DEVICE_ML_FEATURES` | View | Feature store: 29 engineered features from raw telemetry |
| `V_ML_FAILURE_PREDICTIONS` | View | Real-time inference via `MODEL!PREDICT()` |
| `T_ML_PREDICTIONS` | Table | Materialized predictions for high-frequency queries |
| `SV_DEVICE_ANALYTICS` | Semantic View | Agent-accessible unified device health view |

### Key Design Decisions

1. **Label Engineering**: Forward-looking labels from `MAINTENANCE_HISTORY` ensure the model predicts *future* failures
2. **TREND Features**: `CPU_TEMP_TREND`, `MEMORY_TREND`, `ERROR_ACCELERATION` capture degradation velocity—the strongest predictors
3. **Explainability**: `PRIMARY_RISK_FACTOR` column provides human-readable explanations for every prediction
4. **Dual Deployment**: View for real-time, table for performance—both available to downstream consumers

### Integration Points

The predictions are consumed by:
- **Cortex Agent**: Natural language queries against `SV_DEVICE_ANALYTICS`
- **Operational Dashboards**: Risk-level distribution and trending
- **Alerting Systems**: CRITICAL risk devices trigger automated notifications
- **Workforce Management**: Hours-to-failure estimates optimize technician scheduling